# Customer Churn Prediction Using Machine Learning
This notebook follows the full data mining pipeline required for the final project.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

## 2. Load Dataset

In [ ]:
df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head()

## 3. Data Understanding

In [ ]:
print(df.shape)
print(df.info())
print(df['Churn'].value_counts())

## 4. Data Cleaning and Preprocessing

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
if 'customerID' in df.columns:
    df = df.drop(columns=['customerID'])
y = df['Churn'].map({'No':0, 'Yes':1})
X = df.drop(columns=['Churn'])
num_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object','category','bool']).columns.tolist()
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first'))]), cat_cols)
])

## 5. Sampling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)

## 6. PCA Impact: Accuracy, Feature Count, and Time
This section measures the effect of PCA using accuracy, number of features/components, training time, and prediction time.

In [ ]:
def to_dense(X):
    return X.toarray() if hasattr(X, 'toarray') else X

X_processed = preprocessor.fit_transform(X_train)
features_before_pca = X_processed.shape[1]

pca_results = []
pca_feature_counts = []
pca_time_results = []

models_for_pca = {
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42, class_weight='balanced'),
    'SVM': SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced', random_state=42)
}

for model_name, classifier in models_for_pca.items():
    before_model = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', classifier)
    ])

    start = time.perf_counter()
    before_model.fit(X_train, y_train)
    train_before = time.perf_counter() - start

    start = time.perf_counter()
    pred_before = before_model.predict(X_test)
    predict_before = time.perf_counter() - start

    after_model = Pipeline([
        ('preprocessor', preprocessor),
        ('to_dense', FunctionTransformer(to_dense, accept_sparse=True)),
        ('pca', PCA(n_components=0.95, random_state=42)),
        ('classifier', classifier)
    ])

    start = time.perf_counter()
    after_model.fit(X_train, y_train)
    train_after = time.perf_counter() - start

    start = time.perf_counter()
    pred_after = after_model.predict(X_test)
    predict_after = time.perf_counter() - start

    pca_results.append({
        'Model': model_name,
        'Accuracy Before PCA': accuracy_score(y_test, pred_before),
        'Accuracy After PCA': accuracy_score(y_test, pred_after)
    })

    pca_feature_counts.append({
        'Model': model_name,
        'Features Before PCA': features_before_pca,
        'Components After PCA': after_model.named_steps['pca'].n_components_
    })

    pca_time_results.append({
        'Model': model_name,
        'Train Time Before PCA': train_before,
        'Train Time After PCA': train_after,
        'Predict Time Before PCA': predict_before,
        'Predict Time After PCA': predict_after
    })

pca_accuracy_df = pd.DataFrame(pca_results)
pca_features_df = pd.DataFrame(pca_feature_counts)
pca_time_df = pd.DataFrame(pca_time_results)

display(pca_accuracy_df)
display(pca_features_df)
display(pca_time_df)


## 7. Decision Tree and SVM Models

In [ ]:
dt_model = Pipeline([('preprocessor', preprocessor), ('classifier', DecisionTreeClassifier(max_depth=5, random_state=42, class_weight='balanced'))])
svm_model = Pipeline([('preprocessor', preprocessor), ('classifier', SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced', random_state=42))])
dt_model.fit(X_train, y_train)
svm_model.fit(X_train, y_train)

## 8. Evaluation

In [ ]:
def evaluate(name, model):
    pred = model.predict(X_test)
    return {
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred),
        'Recall': recall_score(y_test, pred),
        'F1-score': f1_score(y_test, pred)
    }
results = pd.DataFrame([evaluate('Decision Tree', dt_model), evaluate('SVM', svm_model)])
results

## 9. K-Means Clustering

In [ ]:
X_processed = preprocessor.fit_transform(X_train)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_processed)
print('Clusters:', np.unique(clusters, return_counts=True))